### Create a Small Dataset

In [47]:
import pandas as pd

# Sample dataset with real and fake news
data = {
    'text': [
        'Breaking: New COVID-19 vaccine saves lives!',
        'NASA confirms the discovery of alien life in space.',
        'The government has announced new economic reforms.',
        'Experts say a new species of animal has been discovered.',
        'False: Scientists claim the earth is flat.',
        'Fake News: President signs new law without approval.',
        'Study shows yoga significantly reduces stress levels.',
        'Conspiracy: The moon landing was a hoax.',
        'Breaking: Health experts endorse a new weight loss drug.',
        'Police arrest group of criminals in major operation.',
    ],
    'label': ['real', 'fake', 'real', 'real', 'fake', 'fake', 'real', 'fake', 'real', 'real']
}

df = pd.DataFrame(data)
print(df)


                                                text label
0        Breaking: New COVID-19 vaccine saves lives!  real
1  NASA confirms the discovery of alien life in s...  fake
2  The government has announced new economic refo...  real
3  Experts say a new species of animal has been d...  real
4         False: Scientists claim the earth is flat.  fake
5  Fake News: President signs new law without app...  fake
6  Study shows yoga significantly reduces stress ...  real
7           Conspiracy: The moon landing was a hoax.  fake
8  Breaking: Health experts endorse a new weight ...  real
9  Police arrest group of criminals in major oper...  real


### Preprocess the Text Data
Preprocessing the text is a crucial step. We will:

Convert the text to lowercase.

Remove punctuation and numbers.

Remove stopwords.

Use lemmatization to reduce words to their root form.

In [49]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download stopwords
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize lemmatizer and stopwords
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # Remove punctuation
    text = text.strip()  # Remove leading/trailing spaces
    text = " ".join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])  # Lemmatize and remove stopwords
    return text

df['processed_text'] = df['text'].apply(preprocess_text)
print(df[['text', 'processed_text']])


                                                text  \
0        Breaking: New COVID-19 vaccine saves lives!   
1  NASA confirms the discovery of alien life in s...   
2  The government has announced new economic refo...   
3  Experts say a new species of animal has been d...   
4         False: Scientists claim the earth is flat.   
5  Fake News: President signs new law without app...   
6  Study shows yoga significantly reduces stress ...   
7           Conspiracy: The moon landing was a hoax.   
8  Breaking: Health experts endorse a new weight ...   
9  Police arrest group of criminals in major oper...   

                                      processed_text  
0               breaking new covid vaccine save life  
1           nasa confirms discovery alien life space  
2           government announced new economic reform  
3            expert say new specie animal discovered  
4                   false scientist claim earth flat  
5  fake news president sign new law without approval 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sathi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sathi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Vectorize the Text Data

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize the text data
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))  # Using bigrams and limiting to 5000 features
X = vectorizer.fit_transform(df['processed_text'])

# Encoding the labels (real -> 0, fake -> 1)
y = df['label'].map({'real': 0, 'fake': 1})


###  Split Data into Training and Testing Sets

In [53]:
from sklearn.model_selection import train_test_split

# Split the data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Train a Model (Naive Bayes)

In [55]:
from sklearn.naive_bayes import MultinomialNB

# Initialize and train Naive Bayes model
model = MultinomialNB()
model.fit(X_train, y_train)


MultinomialNB()

### Evaluate the Model

In [57]:
from sklearn.metrics import classification_report, accuracy_score

# Make predictions
y_pred = model.predict(X_test)

# Print classification report
print(classification_report(y_test, y_pred))

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")


              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

Accuracy: 50.00%


C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


###  Tune the Model (Optional)

In [59]:
from sklearn.model_selection import cross_val_score

# Perform cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Cross-validation accuracy scores: {cv_scores}")
print(f"Average cross-validation accuracy: {cv_scores.mean() * 100:.2f}%")


Cross-validation accuracy scores: [0.  0.5 0.5 0.5 0.5]
Average cross-validation accuracy: 40.00%


C:\anaconda\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
import pickle

# Save the model and vectorizer
pickle.dump(model, open("fake_news_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))


In [61]:
import pickle

# Save the model and vectorizer
pickle.dump(model, open("fake_news_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))


In [65]:
# Load the model and vectorizer
model = pickle.load(open("fake_news_model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

# Predict function for new text
def predict_fake_news(news_text):
    processed_text = preprocess_text(news_text)
    text_vector = vectorizer.transform([processed_text])
    prediction = model.predict(text_vector)[0]
    return "Fake News" if prediction == 1 else "Real News"

print("Welcome to the Fake News Detection System!")
user_input = input("Please enter the news text you want to check: ")

# Prediction output
result = predict_fake_news(user_input)
print(f"The news is classified as: {result}")


Welcome to the Fake News Detection System!


Please enter the news text you want to check:  NASA finds new exoplanets in the galaxy


The news is classified as: Real News
